# 09 — SLM-SQL: greedy, fixed-k, CSC and adaptive baselines

Canonical workflow: **09 → 10 → 11 → 12**. Notebook 12 consumes the caches/reports and performs the explicit adaptive policy/merge report stage. This notebook evaluates the author's SFT + GRPO generator with a separate greedy request, one fixed stochastic cache, and real online adaptive sampling. It then runs author CSC for every configured k and on the stopped adaptive candidate sets.

The paper's curves are **pass@k** (oracle), **self_consistency@k** (execution vote) and **SLM-SQL@k** (with author CSC). Greedy is temperature zero; stochastic k=1 is a separate condition. Prompts reconstruct Appendix E semantically, so scores are not claimed as an exact paper reproduction.

Attach BIRD **dev** and, for LoRA pairs, **train**. Set `CSC_TRAIN_DIR` and `CSC_EVAL_DIR` to folders containing `train.json` / `dev.json` and their SQLite subfolders. `BUILD_CSC_TRAIN_CACHE=0` supports dev-only baselines. `CSC_PILOT=0` before first config creation selects the full protocol; use a new config/experiment when changing settings.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
# Run once in an inference session. Enable only if dependencies are not installed.
INSTALL_DEPENDENCIES = os.environ.get('INSTALL_CSC_DEPS', '0') == '1'
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-inference.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run
CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
BUILD_TRAIN_CACHE = os.environ.get('BUILD_CSC_TRAIN_CACHE', '1') == '1'
print({'config': str(CONFIG), 'experiment': cfg['experiment_dir'], 'seed': SEED, 'run_gpu': RUN_GPU})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))

# Pilot defaults to one inference seed; full runs default to three.
_default_seeds = str(SEED) if cfg['data']['evaluation_limit'] else '42,43,44'
EVAL_SEEDS = [int(s) for s in os.environ.get('CSC_EVAL_SEEDS', _default_seeds).split(',')]
if not EVAL_SEEDS or len(set(EVAL_SEEDS)) != len(EVAL_SEEDS) or any(s < 0 for s in EVAL_SEEDS):
    raise ValueError('CSC_EVAL_SEEDS must contain unique nonnegative inference seeds.')
DISPLAY_SEED = EVAL_SEEDS[0]
print('Declared inference seeds:', EVAL_SEEDS)


## Freeze splits and inspect work before using GPU

CPU only: checks database separation, SQLite files and gold SQL compilation, hashes input data and stores schemas. A dev-only preparation can be extended with train/validation later while preserving evaluation candidates. Validation databases are held out from adapter training; the author's upstream model may already have seen BIRD train.


In [ ]:
prepare_flags = [] if BUILD_TRAIN_CACHE else ['--evaluation-only']
run(REPO_DIR, CONFIG, 'prepare', *prepare_flags)
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


## Generator and author CSC sampling matrix

`sampling-baseline` runs fixed generation/execution, greedy, adaptive, then author CSC for each k. Every GPU stage uses a separate subprocess, and completed matching artifacts are reused. Full: k = 1, 2, 4, 8, 16, 32, 64; pilot: k = 1, 2, 4. Author CSC uses 8 merge samples in full / 2 in pilot when there are at least two executable groups.

Adaptive reuses the old Beta or margin stopping logic on the **author SLM-SQL generator** and generates only needed rounds. Its independent draws are not prefix replays of the fixed cache. Default Beta parameters are fixed in config; never select thresholds on final dev EX. Pilot's small cap checks integration and need not show savings.


In [ ]:
K_VALUES = cfg['evaluation']['sampling_k_values']
print({'fixed_k_values': K_VALUES, 'adaptive': cfg['adaptive']})
if RUN_GPU:
    for evaluation_seed in EVAL_SEEDS:
        run(REPO_DIR, CONFIG, 'sampling-baseline', '--seed', str(evaluation_seed))
    run(REPO_DIR, CONFIG, 'report-suite', '--baseline-only', '--seeds', *map(str, EVAL_SEEDS))
else:
    print('GPU stages not started. Set RUN_CSC_GPU=1 after inspecting the sampling matrix in plan.')


## Cache training/validation candidates once

These candidates come from the **training benchmark** partitions, never final dev. No merger call is needed to build LoRA examples: top-2 candidates are checked using train gold execution. Disable this section when only evaluating the author baseline.


In [ ]:
if RUN_GPU and BUILD_TRAIN_CACHE:
    for split in ('train', 'validation'):
        run(REPO_DIR, CONFIG, 'generate', '--split', split, '--seed', str(SEED))
        run(REPO_DIR, CONFIG, 'execute', '--split', split, '--seed', str(SEED))
else:
    print('Training candidate stages not started.')


In [ ]:
report_path = Path(cfg['experiment_dir']) / 'reports' / f'evaluation-seed{DISPLAY_SEED}.sampling.baseline.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    for row in report['table']:
        print(row)
    print('Oracle coverage by k:', {k: value['pass@k'] for k, value in report['summary']['fixed'].items()})
    print('CSV table:', report_path.with_suffix('.csv'))
else:
    print('No completed sampling baseline exists yet; no score is fabricated.')
